# Treinamento: Gatekeeper (Família C - Mag Regional)
**Missão:** Foco agressivo em *Recall* (não perder nada perigoso, assumindo o risco de gerar falsos positivos). Separa a calmaria (Ausência, A, B) de eventos de alerta (C, M, X).

**Metodologia de Split:** *Group-Based Chronological Split* baseado na coluna `REGION_ID` (HARP). O isolamento espacial garante que a assinatura magnética de uma mesma mancha solar não vaze entre treino e teste. Purga temporal (Gap) é desnecessária aqui.

## Setup

In [ ]:
import os
import optuna
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
from optuna.integration import XGBoostPruningCallback
import cupy as cp

from src.py_src.models import GatekeeperModel

In [ ]:
load_dotenv()

REGIONAL_MAG_SLIDED_PATH = os.path.join(os.getenv("SLIDED_PATH"), "mag_regional_slided.parquet")

regional_slided_df = pd.read_parquet(REGIONAL_MAG_SLIDED_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
metadata_cols = ['run_id', 'T_REC_round', 'REGION_ID', 'DATASET_QUERY']

features = [col for col in regional_slided_df.columns if col not in metadata_cols + [target_class, target_flux]]

# Isolamento conceitual das features (X) e do alvo bruto (y)
X = regional_slided_df[features]
y = regional_slided_df[target_class]

In [ ]:
regional_slided_df.head()

## Preparing Data (Group-Based Chronological Split)

In [ ]:
cols_to_keep = features + metadata_cols + [target_class, target_flux]
gatekeeper_pool = regional_slided_df[cols_to_keep].copy()

def group_based_chronological_split(df, time_col, region_col, target_col, flux_col, _train_years, _val_years, _test_years):
    """
    Divide os dados cronologicamente agrupando por Região Ativa (HARP).
    Garante que uma mesma região magnética nunca seja fatiada entre Treino e Validação/Teste.
    """
    df = df.sort_values([region_col, time_col]).reset_index(drop=True).copy()
    
    # 1. Encontra o ano em que a Região Ativa (HARP) surgiu pela primeira vez
    harp_birth = df.groupby(region_col)[time_col].min().dt.year.to_dict()
    df['harp_birth_year'] = df[region_col].map(harp_birth)

    # 2. Atribuição de split baseada no ano de nascimento do HARP
    df['split'] = 'none'
    df.loc[df['harp_birth_year'].isin(_train_years), 'split'] = 'train'
    df.loc[df['harp_birth_year'].isin(_val_years), 'split'] = 'val'
    df.loc[df['harp_birth_year'].isin(_test_years), 'split'] = 'test'

    df = df[df['split'] != 'none'].reset_index(drop=True)

    dict_ = {'x': {}, 'y': {}, 'flux': {}}
    cols_to_drop = [target_col, flux_col, time_col, 'harp_birth_year', 'split'] + metadata_cols

    for split_name in ['train', 'val', 'test']:
        split_df = df[df['split'] == split_name].copy()
        dict_['x'][split_name] = split_df.drop(columns=cols_to_drop, errors='ignore')
        # Gatekeeper Target: >= 3 (Classe C, M, X)
        dict_['y'][split_name] = split_df[target_col].apply(lambda lb: 1 if lb >= 3 else 0)
        dict_['flux'][split_name] = split_df[flux_col]

    return dict_

# Compartilha estritamente a mesma lógica de anos da Família Global X-Ray
# TREINO: Maior parte do Ciclo Solar 24
train_years = [2010, 2011, 2013, 2014, 2015, 2016, 2018, 2019]

# VALIDAÇÃO: 2012 (Alta) e 2017 (Baixa)
val_years = [2012, 2017]

# TESTE: O Ciclo Solar 25 inteiro (Isolado)
test_years = [2020, 2021, 2022, 2023, 2024]

data = group_based_chronological_split(
    df=gatekeeper_pool,
    time_col='T_REC_round',
    region_col='REGION_ID',
    target_col=target_class,
    flux_col=target_flux,
    _train_years=train_years,
    _val_years=val_years,
    _test_years=test_years
)

print(f"Tamanho do Treino: {len(data['x']['train'])} amostras")
print(f"Tamanho da Validação: {len(data['x']['val'])} amostras")
print(f"Tamanho do Teste: {len(data['x']['test'])} amostras")

## Discovery Model
Varredura inicial para descartar features irrelevantes na GPU usando XGBoost Quick Scan.

In [ ]:
discovery_model = GatekeeperModel(
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    }
)

In [ ]:
selected_features = discovery_model.discover_top_features(
    x=data['x']['train'],
    y=data['y']['train'],
    flux_values=data['flux']['train'],
    cumulative_threshold=0.95
)

In [ ]:
selected_features

## Hyperparameter Tuning (Optuna)

In [ ]:
print("Transferindo dados para a VRAM da GPU...")

X_train_filtered = data['x']['train'][selected_features].astype('float32')
X_val_filtered = data['x']['val'][selected_features].astype('float32')

X_train_gpu = cp.array(X_train_filtered.values)
y_train_gpu = cp.array(data['y']['train'].values.astype('float32'))

X_val_gpu = cp.array(X_val_filtered.values)
y_val_gpu = cp.array(data['y']['val'].values.astype('float32'))

print("Transferência concluída. Dados alocados na GPU.")

In [ ]:
def objective(trial):
    neg_count = (data['y']['train'] == 0).sum()
    pos_count = (data['y']['train'] == 1).sum()
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-aucpr')

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'device': 'cuda',

        'early_stopping_rounds': 50,
        'callbacks': [pruning_callback],

        # Este parâmetro é a recomendação oficial do XGBoost para classes altamente desbalanceadas.
        # Restringe a atualização dos pesos da árvore, impedindo que os Falsos Positivos dominem o gradiente.
        'max_delta_step': trial.suggest_int('max_delta_step', 1, 10),

        'scale_pos_weight': trial.suggest_float("scale_pos_weight", imbalance_ratio * 0.5, imbalance_ratio * 2.0),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 25)
    }

    model = GatekeeperModel(params=params, features_to_keep=None)

    model.fit(
        x=X_train_gpu,
        y=y_train_gpu,
        eval_set=[(X_val_gpu, y_val_gpu)],
        verbose=False
    )

    y_pred_proba_raw = model.predict_proba(X_val_gpu)[:, 1]
    y_pred_proba_cpu = y_pred_proba_raw.get() if hasattr(y_pred_proba_raw, 'get') else y_pred_proba_raw

    pr_auc = average_precision_score(data['y']['val'], y_pred_proba_cpu)

    return pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=500)

print(f"\nBest Score (PR AUC): {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'aucpr', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

In [ ]:
final_model = GatekeeperModel(params=study.best_params, features_to_keep=selected_features)
final_model.fit(
    x=data['x']['train'], y=data['y']['train']
)

## Threshold Tuning

In [ ]:
fig = final_model.get_threshold_graph(data['x']['val'], data['y']['val'])
# display(fig)

In [ ]:
optimal_threshold = final_model.optimize_threshold(
    data['x']['val'],
    data['y']['val'],
    target_recall=0.99,
    beta=4.0
)

## Results

In [ ]:
x_test_df = data['x']['test']
y_true = data['y']['test'].values.astype(int)
flux_test = data['flux']['test']

y_prob = final_model.predict_proba(x_test_df)[:, 1]
y_pred = (y_prob >= optimal_threshold).astype(int)

# Nota: Séries temporais de métricas de atividade de flares não estão agrupadas na tabela regional.
# Preencheremos a persistência com zeros estritos para não falhar nas computações métricas.
y_persistence = np.zeros_like(y_true)

In [ ]:
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(final_model.get_classification_report(y_true, y_pred, target_names=['Calmaria (A/B)', 'Alerta (C+)']))

In [ ]:
print("\n--- MÉTRICAS ABRANGENTES ---")
comprehensive_df = final_model.get_comprehensive_metrics(y_true, y_pred, y_prob)
display(comprehensive_df)

In [ ]:
print("\n--- PR-F1 (SKILL SCORE RELATIVO) ---")
# O resultado será influenciado pela persistência base zero, use de maneira informativa.
pr_f1_score = final_model.calculate_prss(y_true, y_pred, y_persistence)
print(f"PR-F1 Score: {pr_f1_score:.4f}")

In [ ]:
print("\n--- ANÁLISE AC/NC (ACTIVITY CHANGE) ---")
ac_nc_df = final_model.analyze_ac_nc_performance(y_true, y_pred, y_persistence)
display(ac_nc_df)

In [ ]:
print("\n--- DISTRIBUIÇÃO DE ERROS POR CLASSE SOLAR ---")
error_dist_df = final_model.analyze_error_distribution(y_true, y_pred, flux_test)
display(error_dist_df)

In [ ]:
print("\n--- ANÁLISE DE FLUXO (ZONAS) ---")
fig_flux, summary_flux = final_model.analyze_flux_errors(y_true, y_pred, flux_test, buffer_limits=[1e-6, 1e-5])
display(summary_flux)
display(fig_flux)

## Features Importance

In [ ]:
features_importance = final_model.get_feature_importance()
display(features_importance.head(15))

## Export

In [ ]:
SAVE_PATH = os.getenv('REGIONAL_MAG_META_MODELS_PATH', os.path.join('models_export', 'regional_mag'))
os.makedirs(SAVE_PATH, exist_ok=True)

final_model.save(os.path.join(SAVE_PATH, 'gatekeeper_v1.joblib'))

print(f"Modelo Gatekeeper (Mag Regional) exportado com sucesso para: {SAVE_PATH}")
print(f"Threshold otimizado embutido: {final_model.threshold:.4f}")
print(f"Total de features retidas: {len(final_model.features_to_keep)}")